# **Interface de Voz e Acessibilidade (UX Assistiva)**
<font size=3>
    
O objetivo desta fase é pegar as coordenadas da Fase 2 e as direções geradas na Fase 3 e entregá-las ao usuário com deficiência visual de forma totalmente auditiva e sensorial. O foco aqui não é a tela visual, mas sim a clareza da **Interface de Voz (VUI)** e a **Acessibilidade Sensorial**.


### **1. Integração com a API `TextToSpeech` (TTS) do Android:**
<font size=3>

A API nativa `TextToSpeech` transforma cadeias de texto em voz sintética no alto-falante ou fones de ouvido do celular.

* **Inicialização e Lifecycle:** O motor de TTS precisa ser inicializado assincronamente e destruído adequadamente quando o app for fechado para evitar vazamento de memória.

* **Configuração do Idioma e Taxa de Fala:** Configurar estritamente para o português do Brasil (`Locale("pt", "BR")`) e ajustar a velocidade de fala (`setSpeechRate`). Pessoas cegas experientes frequentemente utilizam o leitor em velocidades mais altas do que o padrão.

* **Modo de Fila (`QUEUE_FLUSH` vs. `QUEUE_ADD`):**
    * `QUEUE_ADD`: Coloca a nova instrução ao final da fila (útil para detalhes contextuais).
    * `QUEUE_FLUSH`: Interrompe imediatamente qualquer fala em andamento para emitir um alerta urgente (ex: "Atenção: degrau à frente!").

In [ ]:
// ATENÇÃO: o pacote android.* pertence exclusivamente ao sistema operacional e SDK do Android, o Jupyter-Lab irá emitir uma mensagem de erro.
// Assim, faça as adaptações necessárias para rodar no notebook.
import android.content.Context
import android.speech.tts.TextToSpeech
import java.util.Locale

In [ ]:
class VoiceManager(context: Context) : TextToSpeech.OnInitListener {
    private var tts: TextToSpeech? = TextToSpeech(context, this)
    private var isReady: Boolean = false

    override fun onInit(status: Int) {
        if (status == TextToSpeech.SUCCESS) {
            val result = tts?.setLanguage(Locale("pt", "BR"))
            if (result != TextToSpeech.LANG_MISSING_DATA && result != TextToSpeech.LANG_NOT_SUPPORTED) {
                isReady = true
                tts?.setSpeechRate(1.1f) // Ligeiramente mais rápido que o padrão
            }
        }
    }

    fun speak(text: String, interruptPrevious: Boolean = false) {
        if (!isReady) return
        val queueMode = if (interruptPrevious) TextToSpeech.QUEUE_FLUSH else TextToSpeech.QUEUE_ADD
        tts?.speak(text, queueMode, null, "INSTRUCTION_ID_${System.currentTimeMillis()}")
    }

    fun shutdown() {
        tts?.stop()
        tts?.shutdown()
    }
}

### **2. Orientação Espacial Relativa (Sistema do Relógio):**
<font size=3>

Dizer "vire $45^\circ$ a nordeste" não tem utilidade prática para a navegação a pé. Pessoas com deficiência visual se orientam muito melhor com o **Sistema de Ponteiros do Relógio**, tendo o próprio corpo do usuário como centro ($12\text{ horas}$ é sempre a direção em que o peito do usuário está apontado).

* **Conversão do Ângulo em Hora do Relógio:**
    Com base no vetor da bússola do celular (Fase 2) e na direção do próximo nó da rota (Fase 3):
    $$\text{Diferença de Ângulo } (\Delta \theta) = \theta_{\text{destino}} - \theta_{\text{usuario}}$$
    
    $$\text{Hora} = \left( \text{round}\left(\frac{\Delta \theta}{30}\right) \pmod{12} \right) \text{ (se resultado } 0 \to 12\text{)}$$

**Tabela de Mapeamento VUI:**
* **$12 \text{ horas}$ ($[-15^\circ, 15^\circ]$):** "Siga em frente"
* **$1\text{ ou } 2 \text{ horas}$ ($[15^\circ, 75^\circ]$):** "Em frente, ligeiramente à direita"
* **$3 \text{ horas}$ ($[75^\circ, 105^\circ]$):** "À sua direita"
* **$9 \text{ horas}$ ($[-105^\circ, -75^\circ]$):** "À sua esquerda"
* **$6 \text{ horas}$ ($[165^\circ, -165^\circ]$):** "Atrás de você (meia-volta)"

In [ ]:
fun convertToClockSystem(relativeAngleDegrees: Double): String {
    // Normalização do ângulo para [0, 360)
    var normalizedAngle = relativeAngleDegrees % 360
    if (normalizedAngle < 0) normalizedAngle += 360

    val hour = Math.round(normalizedAngle / 30.0).toInt()
    val clockHour = if (hour == 0) 12 else hour

    return when (clockHour) {
        12 -> "às suas 12 horas (em frente)"
        3 -> "às suas 3 horas (à direita)"
        6 -> "às suas 6 horas (atrás de você)"
        9 -> "às suas 9 horas (à esquerda)"
        else -> "às suas $clockHour horas"
    }
}

### **3. Resposta Tátil (Feedback Háptico):**
<font size=3>

O som do fone de ouvido deve ser complementado com vibrações. O feedback háptico serve como confirmação sem a necessidade de interromper a voz do aplicativo.

* **Padrões de Vibração;**
* **Confirmação de Rota / Ponto de Chegada:** Duas vibrações curtas e leves.
* **Alerta de Curva Próxima:** Uma vibração contínua a 2 metros da conversão.
* **Erro de Rota (Desvio do Caminho):** Três vibrações fortes e pulsantes.

In [ ]:
import android.content.Context
import android.os.Build
import android.os.VibrationEffect
import android.os.Vibrator
import android.os.VibratorManager

In [ ]:
fun triggerTurnHapticAlert(context: Context) {
    if (Build.VERSION.SDK_INT >= Build.VERSION_CODES.S) {
        val vibratorManager = context.getSystemService(Context.VIBRATOR_MANAGER_SERVICE) as VibratorManager
        val vibrator = vibratorManager.defaultVibrator
        vibrator.vibrate(VibrationEffect.createOneShot(300, VibrationEffect.DEFAULT_AMPLITUDE))
    } else {
        @Suppress("DEPRECATION")
        val vibrator = context.getSystemService(Context.VIBRATOR_SERVICE) as Vibrator
        @Suppress("DEPRECATION")
        vibrator.vibrate(300)
    }
}

### **4. Diretrizes de Acessibilidade Visual e Leitor de Tela (TalkBack):**
<font size=3>

Para os momentos em que a pessoa com baixa visão interage com a tela do celular (para selecionar o destino, por exemplo), a interface precisa seguir rigidamente o W3C/Android Accessibility Guidelines:

* **Atributo `contentDescription`:** Todos os elementos sem texto visível (como ícones de busca ou botões de voltar) devem obrigatoriamente ter uma descrição em texto para o TalkBack (ex: `android:contentDescription="Botão para selecionar o destino"`).

* **Área de Toque Mínima:** Botões e componentes clicáveis devem ter no mínimo **48x48 dp** de área de toque.

* **Alto Contraste e Tipografia Flexível:** Contraste mínimo de 4.5:1 para texto normal e suporte ao tamanho de fonte configurado nas preferências do sistema do celular.

* **Design Focado na Mão Única:** Telas limpas, sem poluição visual, com botões grandes na parte inferior para facilitar o toque com o polegar enquanto se segura a bengala com a outra mão.


### **Atividade prática:**
<font size=3>
    
Protótipo de Guia de Voz e Vibração:\
Crie uma tela simples em Android contendo um campo de texto para o destino, um botão "Iniciar Navegação" e simule o percurso narrando as etapas.


<details>
<summary>Resposta:</summary>

<font size=2>
<br>
    
```kotlin
class MainActivity : AppCompatActivity() {

    private lateinit var voiceManager: VoiceManager

    override fun onCreate(savedInstanceState: Bundle?) {
        super.onCreate(savedInstanceState)
        setContentView(R.layout.activity_main)

        voiceManager = VoiceManager(this)

        val btnStart = findViewById<Button>(R.id.btnStartNavigation)
        
        btnStart.setOnClickListener {
            simulateNavigationGuide()
        }
    }

    private fun simulateNavigationGuide() {
        // 1. Mensagem de Início
        voiceManager.speak("Navegação iniciada. Destino: Laboratório de TI.", interruptPrevious = true)

        // 2. Simulação de instrução de caminhada com sistema de relógio
        val turnAngle = 90.0 // Curva de 90 graus
        val clockOrientation = convertToClockSystem(turnAngle)
        
        // Simular um atraso de 3 segundos para a próxima instrução
        Handler(Looper.getMainLooper()).postDelayed({
            triggerTurnHapticAlert(this)
            voiceManager.speak("Em 5 passos, vire $clockOrientation. Entrada da sala à direita.")
        }, 3000)
    }

    override fun onDestroy() {
        voiceManager.shutdown()
        super.onDestroy()
    }
}
```

<br>

</details>
